In [1]:
!pip install trl transformers accelerate peft datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00


In [2]:
import json
import os
import gc

# Define your file locations (Update paths if they are in Kaggle input folders)
unannotated_source = "/kaggle/input/datasets/mythreyee1006/report-dataset/dataset_thoracic_unannotated.json"
annotated_source = "/kaggle/input/datasets/mythreyee1006/report-dataset/dataset_thoracic_annotated.json"
output_file = "train_chat.jsonl"

SYSTEM_PROMPT = (
    "You are a clinical assistant. Extract the exact sentence(s) containing "
    "incidental thoracic findings from the report. If none are present, return an empty list."
)

# 1. Load the files
with open(unannotated_source, "r", encoding="utf-8") as f:
    unannotated_reports = json.load(f)["reports"]

with open(annotated_source, "r", encoding="utf-8") as f:
    annotated_reports = json.load(f)["reports"]

# 2. Create a fast lookup map of your annotations using report_id as the key
annotation_lookup = {r["report_id"]: r["annotation"] for r in annotated_reports}

# 3. Build BOTH the SFT chat-format records AND a parallel "structured" record
#    (report_id, free_text, gold) that we'll reuse later for eval. Keeping
#    these aligned means the train/val split is identical between SFT data
#    and eval data.
formatted_records = []
structured_records = []
matched_count = 0

for report in unannotated_reports:
    rid = report["report_id"]
    free_text = report["free_text"]

    if rid in annotation_lookup:
        gold_annotation = annotation_lookup[rid]
        matched_count += 1

        chatml_structure = {
            "report_id": rid,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Report:\n{free_text}"},
                {
                    "role": "assistant",
                    "content": json.dumps({
                        "contains_IF": gold_annotation["contains_IF"],
                        "incidental_sentences": gold_annotation["incidental_sentences"],
                    }),
                },
            ],
        }
        formatted_records.append(chatml_structure)

        structured_records.append({
            "report_id": rid,
            "free_text": free_text,
            "gold": {
                "contains_IF": gold_annotation["contains_IF"],
                "incidental_sentences": gold_annotation["incidental_sentences"],
            },
        })

# 4. Write out the records into a JSONL format (one JSON object per line)
with open(output_file, "w", encoding="utf-8") as f:
    for record in formatted_records:
        f.write(json.dumps(record) + "\n")

print("Mapping complete!")
print(f"Successfully paired {matched_count} annotated files out of {len(unannotated_reports)} total reports.")
print(f"Saved training-ready format to: {output_file}")

Mapping complete!
Successfully paired 1000 annotated files out of 1000 total reports.
Saved training-ready format to: train_chat.jsonl


In [3]:
import torch
import wandb
import numpy as np
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, TrainerCallback, TrainerState, TrainerControl,
    BitsAndBytesConfig,
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split

# --- Config constants (previously undefined) ---
EVAL_BATCH_SIZE = 8
EVAL_STEPS = 50

# --- W&B ---
# Never hardcode API keys in a notebook/script. Set WANDB_API_KEY as an
# environment variable (Kaggle secrets, shell export, etc.) before running.
# If you previously committed a key anywhere, rotate it immediately.
wandb_key = os.environ.get("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")
if wandb_key:
    wandb.login(key=wandb_key)
else:
    print("WANDB_API_KEY not set — skipping wandb.login(); "
          "wandb.init() below may fall back to anonymous/offline mode.")

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
# Left padding is required for correct batched generation with causal LMs.
tokenizer.padding_side = "left"

# --- Load dataset, split ONCE, reuse the same split for SFT text and eval ---
train_structured, val_structured = train_test_split(
    structured_records, test_size=100, random_state=42
)
print(f"Train: {len(train_structured)}, Val: {len(val_structured)}")

train_ids = {r["report_id"] for r in train_structured}
val_ids = {r["report_id"] for r in val_structured}

train_records = [r for r in formatted_records if r["report_id"] in train_ids]
val_records_chat = [r for r in formatted_records if r["report_id"] in val_ids]


def format_chat(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return example


train_dataset = Dataset.from_list(train_records).map(format_chat)

# Token count stats
token_lengths = [len(tokenizer(r["text"])["input_ids"]) for r in train_dataset]
print(f"Token lengths — min: {min(token_lengths)}, max: {max(token_lengths)}, "
      f"mean: {np.mean(token_lengths):.0f}, p95: {np.percentile(token_lengths, 95):.0f}")

# `val_structured` (report_id, free_text, gold) is what run_eval() consumes below.

WANDB_API_KEY not set — skipping wandb.login(); wandb.init() below may fall back to anonymous/offline mode.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train: 900, Val: 100


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Token lengths — min: 87, max: 1040, mean: 383, p95: 637


In [4]:
import re


def parse_output(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', raw_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


def build_messages(record, few_shot_pool=None, n_shot=0):
    """Build the chat message list for a single eval record.

    record: dict with at least "free_text" (and "gold" for few-shot examples).
    few_shot_pool: list of records with the same shape, used to draw few-shot
        examples from (should NOT overlap with the eval set).
    n_shot: number of few-shot examples to prepend.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if few_shot_pool and n_shot > 0:
        for ex in few_shot_pool[:n_shot]:
            messages.append({"role": "user", "content": f"Report:\n{ex['free_text']}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps({
                    "contains_IF": ex["gold"]["contains_IF"],
                    "incidental_sentences": ex["gold"]["incidental_sentences"],
                }),
            })

    messages.append({"role": "user", "content": f"Report:\n{record['free_text']}"})
    return messages


def run_eval(model, tokenizer, records, few_shot_pool=None, n_shot=0, n=None,
             desc="", batch_size=EVAL_BATCH_SIZE):
    import transformers
    transformers.logging.set_verbosity_error()
    model.eval()

    subset = records[:n] if n else records
    total_tp = total_fp = total_fn = 0
    neg_scores, pos_scores = [], []

    for start in range(0, len(subset), batch_size):
        batch_records = subset[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                build_messages(r, few_shot_pool, n_shot),
                tokenize=False, add_generation_prompt=True
            )
            for r in batch_records
        ]

        inputs = tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        for i, record in enumerate(batch_records):
            generated = output_ids[i][prompt_len:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            parsed = parse_output(raw)

            gold_set = set(record["gold"]["incidental_sentences"])
            pred_set = set(parsed.get("incidental_sentences", [])) if parsed else set()

            if len(gold_set) == 0:
                neg_scores.append(1.0 if len(pred_set) == 0 else 0.0)
            else:
                tp = len(gold_set & pred_set)
                fp = len(pred_set - gold_set)
                fn = len(gold_set - pred_set)
                total_tp += tp
                total_fp += fp
                total_fn += fn
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                pos_scores.append(f1)

    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    macro_f1 = (avg_neg + avg_pos) / 2 if (neg_scores and pos_scores) else (avg_neg or avg_pos)

    n_neg, n_pos = len(neg_scores), len(pos_scores)
    weighted_f1 = (n_neg * avg_neg + n_pos * avg_pos) / (n_neg + n_pos) if (n_neg + n_pos) > 0 else 0.0

    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0

    model.train()
    return {
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
    }


class MacroF1EarlyStoppingCallback(TrainerCallback):
    def __init__(self, eval_records, tokenizer, run_name, eval_steps=EVAL_STEPS, patience=3):
        self.eval_records = eval_records
        self.tokenizer = tokenizer
        self.run_name = run_name
        self.eval_steps = eval_steps
        self.patience = patience
        self.best_f1 = -1.0
        self.no_improve = 0
        self.best_step = 0
        self.best_metrics = None
        self.best_ckpt_dir = f"./qlora_best_{run_name}"

    def on_step_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        if state.global_step % self.eval_steps != 0 or state.global_step == 0:
            return control

        model = kwargs["model"]
        metrics = run_eval(model, self.tokenizer, self.eval_records, n=100, desc="val")

        print(f"\nStep {state.global_step} | Macro F1: {metrics['macro_f1']:.4f} | "
              f"Micro F1: {metrics['micro_f1']:.4f} | Weighted F1: {metrics['weighted_f1']:.4f} | "
              f"Best: {self.best_f1:.4f}")
        wandb.log({
            "val_macro_f1": metrics["macro_f1"],
            "val_micro_f1": metrics["micro_f1"],
            "val_weighted_f1": metrics["weighted_f1"],
            "step": state.global_step,
        })

        if metrics["macro_f1"] > self.best_f1:
            self.best_f1 = metrics["macro_f1"]
            self.best_step = state.global_step
            self.best_metrics = metrics
            self.no_improve = 0
            model.save_pretrained(self.best_ckpt_dir)
            self.tokenizer.save_pretrained(self.best_ckpt_dir)
            print(f"  New best saved -> {self.best_ckpt_dir}")
        else:
            self.no_improve += 1
            print(f"  No improvement ({self.no_improve}/{self.patience})")

        if self.no_improve >= self.patience:
            print(f"\nEarly stopping at step {state.global_step}. "
                  f"Best step {self.best_step}, macro F1={self.best_f1:.4f}")
            control.should_training_stop = True

        return control

In [5]:
if wandb_key:
    wandb.login(key=wandb_key)
else:
    os.environ["WANDB_MODE"] = "disabled"  # or "offline" to log locally without a key
    print("WANDB_API_KEY not set — running with wandb disabled.")

WANDB_API_KEY not set — running with wandb disabled.


In [6]:
# Hyperparameter grid
lora_configs = [
    {"r": 32, "lora_alpha": 64},
]

sweep_results = []

# int8 quantization config, shared across sweep runs. This is what actually
# makes the loaded model comparable to a GGUF Q8_0 QLoRA baseline — the
# previous `torch_dtype=torch.int8` line did NOT do real quantization, it
# just cast raw weight values into int8 range with no scale/zero-point.
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

for config in lora_configs:
    run_name = f"r{config['r']}_alpha{config['lora_alpha']}"
    print(f"\n{'=' * 50}")
    print(f"Starting run: {run_name}")
    print(f"{'=' * 50}")

    wandb.init(
        project="incidental-findings-finetuning",
        name=run_name,
        config=config,
        reinit=True,
    )

    # Reload a fresh base model, quantized to int8, for each run.
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map={"": 0},
    )
    model = prepare_model_for_kbit_training(model)

    peft_config = LoraConfig(
        r=config["r"],
        lora_alpha=config["lora_alpha"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    training_args = SFTConfig(
        output_dir=f"./qwen_lora_{run_name}",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        num_train_epochs=100,        # high ceiling, early stopping will trigger
        save_strategy="steps",
        save_steps=50,
        eval_strategy="no",          # we handle eval in the callback
        lr_scheduler_type="cosine",
        warmup_steps=10,
        fp16=False,                  # base weights are int8; LoRA adapters stay fp32/bf16
        bf16=False,
        report_to="wandb",
        dataset_text_field="text",
        gradient_checkpointing=True,
    )

    callback = MacroF1EarlyStoppingCallback(
        eval_records=val_structured,
        tokenizer=tokenizer,
        run_name=run_name,
        eval_steps=50,
        patience=3,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_args,
        callbacks=[callback],
    )

    trainer.train()

    sweep_results.append({
        "run": run_name,
        "r": config["r"],
        "lora_alpha": config["lora_alpha"],
        "best_f1": callback.best_f1,
        "best_step": callback.best_step,
    })

    wandb.finish()

    # Free GPU memory before the next sweep iteration.
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

# Summary
print("\n" + "=" * 50)
print("Sweep Summary:")
print("=" * 50)
for r in sorted(sweep_results, key=lambda x: x["best_f1"], reverse=True):
    print(f"  {r['run']:20s} | Best F1: {r['best_f1']:.4f} | Step: {r['best_step']}")


Starting run: r32_alpha64


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

...


Step 650 | Macro F1: 0.6267 | Micro F1: 0.5000 | Weighted F1: 0.5625 | Best: 0.6468
  No improvement (1/3)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



Step 700 | Macro F1: 0.5819 | Micro F1: 0.5097 | Weighted F1: 0.5377 | Best: 0.6468
  No improvement (2/3)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



Step 750 | Macro F1: 0.6401 | Micro F1: 0.5558 | Weighted F1: 0.6046 | Best: 0.6468
  No improvement (3/3)

Early stopping at step 750. Best step 600, macro F1=0.6468

Sweep Summary:
  r32_alpha64          | Best F1: 0.6468 | Step: 600


In [7]:
# %% [code]
# Standalone evaluation script.
# Run this AFTER training — it does not train anything, it just loads a
# saved LoRA checkpoint and scores it against the held-out test set.

import json
import re
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --- Config: point this at whichever checkpoint you want to evaluate ---
checkpoint_dir = "./qlora_best_r32_alpha64"   # <- change to the run you want to evaluate

test_unannotated_source = "/kaggle/input/datasets/mythreyee1006/testdataset-thoractic-ct/test_dataset_thoracic_unannotated.json"
test_annotated_source = "/kaggle/input/datasets/mythreyee1006/testdataset-thoractic-ct/test_dataset_thoracic_annotated.json"

SYSTEM_PROMPT = (
    "You are a clinical assistant. Extract the exact sentence(s) containing "
    "incidental thoracic findings from the report. If none are present, return an empty list."
)

EVAL_BATCH_SIZE = 8

# %% [code]
# --- Load test set ---
with open(test_unannotated_source, "r", encoding="utf-8") as f:
    test_unannotated_reports = json.load(f)["reports"]

with open(test_annotated_source, "r", encoding="utf-8") as f:
    test_annotated_reports = json.load(f)["reports"]

test_annotation_lookup = {r["report_id"]: r["annotation"] for r in test_annotated_reports}

test_structured_records = []
for report in test_unannotated_reports:
    rid = report["report_id"]
    if rid in test_annotation_lookup:
        gold_annotation = test_annotation_lookup[rid]
        test_structured_records.append({
            "report_id": rid,
            "free_text": report["free_text"],
            "gold": {
                "contains_IF": gold_annotation["contains_IF"],
                "incidental_sentences": gold_annotation["incidental_sentences"],
            },
        })

print(f"Loaded {len(test_structured_records)} test records "
      f"(of {len(test_unannotated_reports)} total, matched to annotations).")

# %% [code]
# --- Load tokenizer + quantized base model + LoRA adapter ---
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for correct batched generation

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
)
model = PeftModel.from_pretrained(base_model, checkpoint_dir)
model.eval()

# %% [code]
# --- Eval helpers ---

def parse_output(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', raw_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


def build_messages(record, few_shot_pool=None, n_shot=0):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if few_shot_pool and n_shot > 0:
        for ex in few_shot_pool[:n_shot]:
            messages.append({"role": "user", "content": f"Report:\n{ex['free_text']}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps({
                    "contains_IF": ex["gold"]["contains_IF"],
                    "incidental_sentences": ex["gold"]["incidental_sentences"],
                }),
            })

    messages.append({"role": "user", "content": f"Report:\n{record['free_text']}"})
    return messages


def run_eval(model, tokenizer, records, few_shot_pool=None, n_shot=0, n=None,
             batch_size=EVAL_BATCH_SIZE):
    import transformers
    transformers.logging.set_verbosity_error()
    model.eval()

    subset = records[:n] if n else records
    total_tp = total_fp = total_fn = 0
    neg_scores, pos_scores = [], []

    for start in range(0, len(subset), batch_size):
        batch_records = subset[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                build_messages(r, few_shot_pool, n_shot),
                tokenize=False, add_generation_prompt=True
            )
            for r in batch_records
        ]

        inputs = tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        for i, record in enumerate(batch_records):
            generated = output_ids[i][prompt_len:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            parsed = parse_output(raw)

            gold_set = set(record["gold"]["incidental_sentences"])
            pred_set = set(parsed.get("incidental_sentences", [])) if parsed else set()

            if len(gold_set) == 0:
                neg_scores.append(1.0 if len(pred_set) == 0 else 0.0)
            else:
                tp = len(gold_set & pred_set)
                fp = len(pred_set - gold_set)
                fn = len(gold_set - pred_set)
                total_tp += tp
                total_fp += fp
                total_fn += fn
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                pos_scores.append(f1)

    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    macro_f1 = (avg_neg + avg_pos) / 2 if (neg_scores and pos_scores) else (avg_neg or avg_pos)

    n_neg, n_pos = len(neg_scores), len(pos_scores)
    weighted_f1 = (n_neg * avg_neg + n_pos * avg_pos) / (n_neg + n_pos) if (n_neg + n_pos) > 0 else 0.0

    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0

    return {
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
        "n_negative": n_neg,
        "n_positive": n_pos,
    }

# %% [code]
# --- Run it ---
metrics = run_eval(model, tokenizer, test_structured_records, n=None)

print(f"\nTest set results ({len(test_structured_records)} records — "
      f"{metrics['n_negative']} no-finding, {metrics['n_positive']} with-finding):")
print(f"  Macro F1:    {metrics['macro_f1']:.4f}")
print(f"  Micro F1:    {metrics['micro_f1']:.4f}")
print(f"  Weighted F1: {metrics['weighted_f1']:.4f}")

del model, base_model
gc.collect()
torch.cuda.empty_cache()

Loaded 100 test records (of 100 total, matched to annotations).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



Test set results (100 records — 25 no-finding, 75 with-finding):
  Macro F1:    0.6011
  Micro F1:    0.4498
  Weighted F1: 0.5217
